# 03 Regression: Productivity and Digital Dependence Scores


## Objective

补齐回归调参和 MSE 指标，同时运行两个目标：`productivity_score` 与 `digital_dependence_score`。如果生产力预测仍然弱，将作为负结果诚实记录。


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().resolve().parent if (Path.cwd().resolve().parent / "src").exists() else PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT / "src"))
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

from config import FIGURES_DIR, RANDOM_STATE, REGRESSION_BACKUP_TARGET, REGRESSION_TARGET
from data_utils import ensure_project_dirs, load_processed_dataset
from feature_engineering import make_feature_target
from model_utils import run_regression_suite
from visualization import (
    plot_feature_importance,
    plot_regression_predictions,
    plot_regression_residuals,
    plot_regression_target_comparison,
)

np.random.seed(RANDOM_STATE)
ensure_project_dirs()


PROJECT_ROOT = C:\Users\qintian\Desktop\大数据\Big-Data-Homework\期末考查报告_数字生活方式分析


## Regression Feature Checks

两个回归目标都排除目标列、另一个 outcome 目标变量、`high_risk_flag` 和心理状态结果变量。


In [2]:
df = load_processed_dataset(fallback_to_raw=True)
for target in [REGRESSION_TARGET, REGRESSION_BACKUP_TARGET]:
    X, y = make_feature_target(df, task="regression", target_column=target)
    assert target not in X.columns
    print(f"Target: {target}; feature matrix shape: {X.shape}; target mean={y.mean():.4f}; std={y.std():.4f}")


Target: productivity_score; feature matrix shape: (3500, 22); target mean=65.2993; std=9.6647
Target: digital_dependence_score; feature matrix shape: (3500, 22); target mean=36.6842; std=14.1155


## Tuned Regression Experiments

每个目标分别对 Ridge、Random Forest、Gradient Boosting 等模型做 5 折交叉验证调参，并输出 R²、MSE、RMSE、MAE。


In [3]:
regression_suite = run_regression_suite(df)
productivity_result = regression_suite["productivity"]
dependence_result = regression_suite["digital_dependence"]
comparison = regression_suite["comparison"]

display(productivity_result["metrics"])
display(dependence_result["metrics"])
display(comparison)


,model,target,cv_best_r2,cv_best_mse,cv_best_mae,n_train,n_test,feature_count,best_params,mae,mse,rmse,r2
0,gradient_boosting,productivity_score,0.011926,94.955100,7.614592,2625,875,22,"{'model__n_estimators': 100, 'model__max_depth...",7.167096,85.203131,9.230554,-0.004064
1,random_forest,productivity_score,0.007174,95.415587,7.632957,2625,875,22,"{'model__n_estimators': 400, 'model__min_sampl...",7.208846,85.920799,9.269347,-0.012521
2,ridge,productivity_score,0.004320,95.672501,7.662626,2625,875,22,{'model__alpha': 100},7.174703,85.588430,9.251401,-0.008605
3,linear_regression,productivity_score,0.000183,96.073039,7.687985,2625,875,22,{},7.199923,85.823216,9.264082,-0.011371


,model,target,cv_best_r2,cv_best_mse,cv_best_mae,n_train,n_test,feature_count,best_params,mae,mse,rmse,r2
0,gradient_boosting,digital_dependence_score,0.980448,3.897547,1.126590,2625,875,22,"{'model__n_estimators': 200, 'model__max_depth...",0.998244,3.147071,1.773999,0.983901
1,ridge,digital_dependence_score,0.969168,6.046784,0.896467,2625,875,22,{'model__alpha': 0.1},0.765191,3.654067,1.911561,0.981308
2,linear_regression,digital_dependence_score,0.969168,6.046825,0.896324,2625,875,22,{},0.765067,3.653820,1.911497,0.981309
3,random_forest,digital_dependence_score,0.967974,6.348078,1.485639,2625,875,22,"{'model__n_estimators': 400, 'model__min_sampl...",1.268012,4.387505,2.094637,0.977556


,model,target,cv_best_r2,cv_best_mse,cv_best_mae,n_train,n_test,feature_count,best_params,mae,mse,rmse,r2
0,gradient_boosting,productivity_score,0.011926,94.955100,7.614592,2625,875,22,"{'model__n_estimators': 100, 'model__max_depth...",7.167096,85.203131,9.230554,-0.004064
1,gradient_boosting,digital_dependence_score,0.980448,3.897547,1.126590,2625,875,22,"{'model__n_estimators': 200, 'model__max_depth...",0.998244,3.147071,1.773999,0.983901


## Regression Figures

生产力预测和数字依赖预测分别保存真实值-预测值图、残差图和置换重要性图；目标对比图用于判断哪个回归目标更适合报告主线。


In [4]:
plot_regression_predictions(
    productivity_result["y_test"],
    productivity_result["y_pred"],
    FIGURES_DIR / "regression_productivity_observed_vs_predicted.png",
)
plot_regression_residuals(
    productivity_result["y_test"],
    productivity_result["y_pred"],
    FIGURES_DIR / "regression_productivity_residuals.png",
)
plot_feature_importance(
    productivity_result["feature_importance"],
    FIGURES_DIR / "regression_productivity_permutation_importance.png",
    title="Productivity Regression Permutation Importance",
)

plot_regression_predictions(
    dependence_result["y_test"],
    dependence_result["y_pred"],
    FIGURES_DIR / "regression_digital_dependence_observed_vs_predicted.png",
)
plot_regression_residuals(
    dependence_result["y_test"],
    dependence_result["y_pred"],
    FIGURES_DIR / "regression_digital_dependence_residuals.png",
)
plot_feature_importance(
    dependence_result["feature_importance"],
    FIGURES_DIR / "regression_digital_dependence_permutation_importance.png",
    title="Digital Dependence Regression Permutation Importance",
)
plot_regression_target_comparison(comparison, FIGURES_DIR / "regression_target_comparison.png")


WindowsPath('C:/Users/qintian/Desktop/大数据/Big-Data-Homework/期末考查报告_数字生活方式分析/figures/regression_target_comparison.png')